# Полная сборка корпуса Pinega → PEZ → Hugging Face

Ноутбук автоматически открывает каталог **Fulltexts**, находит ссылки **Annotated text**, обходит страницы записей, извлекает сегменты и скачивает соответствующие WAV. Для каждого сегмента создаются `.wav`, `.txt` и `_orig.txt`. Полные транскрипции записей отдельно **не сохраняются**.

Результат загружается в приватный репозиторий Hugging Face Dataset. Перед публикацией проверьте условия использования аудио и транскрипций на сайте-источнике.

**Важно:** в Colab Secrets создайте `HF_TOKEN` с правами Write. Укажите свой `HF_REPO_ID` ниже.

In [1]:
!pip -q install requests beautifulsoup4 urllib3 huggingface_hub playwright nest_asyncio
!playwright install chromium


191.1 MiB [] 0% 393.6s191.1 MiB [] 0% 25.5s191.1 MiB [] 0% 16.5s191.1 MiB [] 0% 14.5s191.1 MiB [] 0% 6.8s191.1 MiB [] 1% 4.2s191.1 MiB [] 2% 3.5s191.1 MiB [] 3% 3.1s191.1 MiB [] 4% 3.2s191.1 MiB [] 4% 3.5s191.1 MiB [] 4% 3.6s191.1 MiB [] 4% 3.7s191.1 MiB [] 5% 3.9s191.1 MiB [] 6% 3.5s191.1 MiB [] 7% 3.2s191.1 MiB [] 8% 2.9s191.1 MiB [] 9% 2.7s191.1 MiB [] 10% 2.6s191.1 MiB [] 11% 2.5s191.1 MiB [] 11% 2.4s191.1 MiB [] 12% 2.3s191.1 MiB [] 13% 2.4s191.1 MiB [] 13% 2.5s191.1 MiB [] 13% 2.6s191.1 MiB [] 13% 2.7s191.1 MiB [] 14% 2.6s191.1 MiB [] 15% 2.5s191.1 MiB [] 16% 2.4s191.1 MiB [] 17% 2.4s191.1 MiB [] 18% 2.3s191.1 MiB [] 19% 2.2s191.1 MiB [] 20% 2.1s191.1 MiB [] 21% 2.1s191.1 MiB [] 21% 2.2s191.1 MiB [] 21% 2.3s191.1 MiB [] 22% 2.3s191.1 MiB [] 23% 2.2s191.1 MiB [] 24% 2.1s191.1 MiB [] 25% 2.0s191.1 MiB [] 27% 1.9s191.1 MiB [] 28% 1.9s191.1 MiB [] 29% 1.8s191.1 MiB [] 31% 1.7s191.1 MiB [] 32% 1.6s191.1 MiB [] 34% 1.6s191.1 MiB [] 35% 1.5s191.1 MiB [] 36% 1.5s191.1 MiB [] 37% 1.4s191.

## 1. Подключение Google Drive и настройки

In [2]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [14]:
# Каталог Fulltexts (hash-route)
INDEX_URL = 'https://lingconlab.ru/pinega/#!/fulltexts'

# Укажите свой репозиторий HF в формате username/dataset-name
HF_REPO_ID = 'katykool/NortDialect_benchmark'
HF_PRIVATE = True

# Данные сохраняются на Google Drive между сессиями Colab
OUTPUT_DIR = '/content/drive/MyDrive/pinega_dataset'
DELAY_BETWEEN_AUDIO_DOWNLOADS = 0.35

# Ограничение для тестового прогона. None = все найденные страницы.
MAX_RECORDINGS = 1


In [8]:
import sys
import subprocess

print("Устанавливаю системные библиотеки для Chromium...")

subprocess.run(
    [sys.executable, "-m", "playwright", "install-deps", "chromium"],
    check=True
)

print("Устанавливаю Chromium...")

subprocess.run(
    [sys.executable, "-m", "playwright", "install", "chromium"],
    check=True
)

print("Готово!")

Устанавливаю системные библиотеки для Chromium...
Устанавливаю Chromium...
Готово!


In [9]:
import asyncio
from playwright.async_api import async_playwright

async def test_chromium():
    async with async_playwright() as p:
        browser = await p.chromium.launch(
            headless=True,
            args=["--no-sandbox"]
        )

        page = await browser.new_page()
        await page.goto(
            "https://lingconlab.ru/pinega/#!/fulltexts",
            wait_until="domcontentloaded",
            timeout=120000
        )

        print("Chromium успешно запущен!")
        print("Заголовок страницы:", await page.title())

        await browser.close()

await test_chromium()

Chromium успешно запущен!
Заголовок страницы: Middle Pinega Corpus


In [23]:
CORPUS_URLS = {
    "pyoza": "https://lingconlab.ru/pyoza/#!/fulltexts",
    "veegora": "https://lingconlab.ru/veegora/#!/fulltexts",
    "pinega": "https://lingconlab.ru/pinega/#!/fulltexts",
    "keba": "https://lingconlab.ru/keba/#!/fulltexts",
    "vaduga": "https://lingconlab.ru/vaduga/#!/fulltexts",
    "dvina": "https://lingconlab.ru/dvina/#!/fulltexts",
    "tserkovnoe": "https://lingconlab.ru/tserkovnoe/#!/fulltexts",
}

## 2. Авторизация Hugging Face

In [5]:
from google.colab import userdata
from huggingface_hub import HfApi, login

hf_token = userdata.get('HF_TOKEN')
if not hf_token:
    raise RuntimeError('Добавьте HF_TOKEN в Colab → Secrets и включите доступ для ноутбука.')

login(token=hf_token, add_to_git_credential=False)
api = HfApi(token=hf_token)
api.create_repo(repo_id=HF_REPO_ID, repo_type='dataset', private=HF_PRIVATE, exist_ok=True)
print(f'Репозиторий готов: https://huggingface.co/datasets/{HF_REPO_ID}')


Репозиторий готов: https://huggingface.co/datasets/katykool/NortDialect_benchmark


## 3. Получить список Annotated text

Поскольку `#!/fulltexts` — hash-маршрут JavaScript-приложения, список извлекается через браузер Chromium, а не обычным HTML-запросом. Код ищет ссылки на `files_html/*.html`, связанные с надписью `Annotated text`.

In [24]:
import asyncio
import nest_asyncio
from playwright.async_api import async_playwright

nest_asyncio.apply()

async def discover_annotated_pages(index_url):
    urls = set()
    anchors = []

    async with async_playwright() as p:
        browser = await p.chromium.launch(
            headless=True,
            args=["--no-sandbox"]
        )

        page = await browser.new_page()

        print(f"Открываю каталог: {index_url}")

        await page.goto(
            index_url,
            wait_until="domcontentloaded",
            timeout=120000
        )

        # Даём SPA время загрузить список записей
        try:
            await page.wait_for_load_state(
                "networkidle",
                timeout=30000
            )
        except Exception:
            pass

        await page.wait_for_timeout(5000)

        # Собираем все ссылки на HTML-файлы
        anchor_data = await page.locator("a").evaluate_all("""
            elements => elements.map(a => ({
                text: (a.innerText || a.textContent || "").trim(),
                href: a.href
            }))
        """)

        anchors.extend(anchor_data)

        for item in anchor_data:
            href = item.get("href", "")

            if not href:
                continue

            href = href.split("#")[0].split("?")[0]

            if (
                "files_html/" in href
                and href.lower().endswith(".html")
            ):
                urls.add(href)

        await browser.close()

    urls = sorted(urls)

    print(f"Найдено HTML-страниц: {len(urls)}")

    return urls, anchors


# ВАЖНО: в Colab используем await, а не run_until_complete
CORPUS_URLS = {
    "pyoza": "https://lingconlab.ru/pyoza/#!/fulltexts",
    "veegora": "https://lingconlab.ru/veegora/#!/fulltexts",
    "pinega": "https://lingconlab.ru/pinega/#!/fulltexts",
    "keba": "https://lingconlab.ru/keba/#!/fulltexts",
    "vaduga": "https://lingconlab.ru/vaduga/#!/fulltexts",
    "dvina": "https://lingconlab.ru/dvina/#!/fulltexts",
    "tserkovnoe": "https://lingconlab.ru/tserkovnoe/#!/fulltexts",
}

corpus_pages = {}

for corpus_name, index_url in CORPUS_URLS.items():
    print(f"\n{'=' * 60}")
    print(f"КОРПУС: {corpus_name}")
    print(f"{'=' * 60}")

    page_urls, discovered_anchor_dump = await discover_annotated_pages(
        index_url
    )

    corpus_pages[corpus_name] = page_urls

    print(f"Записей в корпусе {corpus_name}: {len(page_urls)}")

    if not page_urls:
        print("Внимание: HTML-страницы не найдены.")

print("\nИТОГО:")
for corpus_name, urls in corpus_pages.items():
    print(f"{corpus_name}: {len(urls)} записей")

print(
    "\nВсего HTML-страниц:",
    sum(len(urls) for urls in corpus_pages.values())
)

print("\nПримеры найденных страниц:")
for url in page_urls[:10]:
    print(url)

if not page_urls:
    print("\nСсылки не найдены.")
    print("Первые обнаруженные ссылки:")
    for item in discovered_anchor_dump[:30]:
        print(item)


КОРПУС: pyoza
Открываю каталог: https://lingconlab.ru/pyoza/#!/fulltexts
Найдено HTML-страниц: 9
Записей в корпусе pyoza: 9

КОРПУС: veegora
Открываю каталог: https://lingconlab.ru/veegora/#!/fulltexts


CancelledError: 

## 4. Сборщик сегментов

Следующий код — тот же парсер страницы записи, но он запускается отдельно для каждой найденной страницы. WAV-файлы скачиваются по исходным ссылкам из HTML. Псевдонимы говорящих стабильны между запусками благодаря приватному ключу `.speaker_salt`.

In [ ]:
import sys
from pathlib import Path

# Notebook and collector.py must be in this project folder on Google Drive.
PROJECT_DIR = Path('/content/drive/MyDrive/pinega_pez_project')
if not PROJECT_DIR.exists():
    raise FileNotFoundError(f'Project folder not found: {PROJECT_DIR}')
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

import pinega_pez_collector as collector
print(f'[setup] Imported collector: {Path(collector.__file__).resolve()}')


In [28]:
import requests
from bs4 import BeautifulSoup, NavigableString, Tag
from collections import Counter
from urllib.parse import urljoin
from tqdm.auto import tqdm
import re


# ============================================================
# НАСТРОЙКИ
# ============================================================

PREVIEW_LIMIT = 5
RECORDINGS_PER_CORPUS = 3
REQUEST_TIMEOUT = 60

session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0 (compatible; LingConLabPreview/1.0)"
})


# ============================================================
# ИЗВЛЕЧЕНИЕ СЕГМЕНТОВ ИЗ HTML
# ============================================================

def preview_segments(url, limit=PREVIEW_LIMIT):

    response = session.get(url, timeout=REQUEST_TIMEOUT)
    response.raise_for_status()
    response.encoding = response.apparent_encoding or "utf-8"

    soup = BeautifulSoup(response.text, "html.parser")
    body = soup.body or soup

    audio_nodes = body.find_all("audio")

    segments = []
    pending_text = []
    current_speaker = "UNKNOWN"

    # Обход DOM с прогрессом по WAV
    audio_counter = tqdm(
        total=len(audio_nodes),
        desc=f"WAV {url.split('/')[-1]}",
        unit="сегм.",
        dynamic_ncols=True,
        leave=False
    )

    for node in body.descendants:

        # ----------------------------------------------------
        # Подписи говорящих
        # ----------------------------------------------------

        if isinstance(node, Tag) and node.name == "b":

            label = node.get_text(" ", strip=True).rstrip(":").strip()

            if label:
                current_speaker = label
                pending_text = []

        # ----------------------------------------------------
        # Текст между аудиофрагментами
        # ----------------------------------------------------

        elif isinstance(node, NavigableString):

            parent = node.parent

            if parent and parent.name in [
                "b", "script", "style", "audio", "source"
            ]:
                continue

            pending_text.append(str(node))

        # ----------------------------------------------------
        # Аудиофрагменты
        # ----------------------------------------------------

        elif isinstance(node, Tag) and node.name == "audio":

            audio_id = node.get("id", "")

            audio_src = node.get("src")

            # Если src находится внутри <source>
            if not audio_src:

                source = node.find("source")

                if source:
                    audio_src = source.get("src")

            text = re.sub(
                r"\s+",
                " ",
                "".join(pending_text)
            ).strip()

            if audio_id:

                segments.append({
                    "audio_id": audio_id,
                    "audio_src": (
                        urljoin(url, audio_src)
                        if audio_src else None
                    ),
                    "speaker": current_speaker,
                    "text": text
                })

            pending_text = []

            audio_counter.update(1)

    audio_counter.close()

    # ========================================================
    # СТАТИСТИКА ЗАПИСИ
    # ========================================================

    print("\nЗапись:", url.split("/")[-1])
    print("Всего audio в HTML:", len(audio_nodes))
    print("Извлечено сегментов:", len(segments))
    print("Говорящие:", Counter(s["speaker"] for s in segments))

    print("\nПервые сегменты:")

    for i, segment in enumerate(segments[:limit], 1):

        print(f"\n--- Сегмент {i} ---")
        print("ID:", segment["audio_id"])
        print("Speaker:", segment["speaker"])
        print("WAV:", segment["audio_src"])
        print("TEXT:", segment["text"][:500] or "[ПУСТОЙ ТЕКСТ]")

    empty = sum(not s["text"] for s in segments)
    missing_audio = sum(not s["audio_src"] for s in segments)

    print(f"\nПустых текстовых сегментов: {empty}")
    print(f"Сегментов без ссылки на WAV: {missing_audio}")

    return segments


# ============================================================
# ПРОВЕРКА ВСЕХ СЕМИ КОРПУСОВ
# ============================================================

preview_results = {}

total_recordings = sum(
    min(len(urls), RECORDINGS_PER_CORPUS)
    for urls in corpus_pages.values()
)

recording_progress = tqdm(
    total=total_recordings,
    desc="Все записи",
    unit="запись",
    dynamic_ncols=True
)

for corpus_name, urls in corpus_pages.items():

    print("\n" + "=" * 70)
    print("КОРПУС:", corpus_name.upper())
    print("=" * 70)

    preview_results[corpus_name] = {}

    selected_urls = urls[:RECORDINGS_PER_CORPUS]

    for url in selected_urls:

        try:

            segments = preview_segments(url, limit=PREVIEW_LIMIT)

            preview_results[corpus_name][url] = segments

        except Exception as exc:

            print(f"\n[ERROR] {url}")
            print(type(exc).__name__, str(exc))

            preview_results[corpus_name][url] = []

        finally:
            recording_progress.update(1)

recording_progress.close()


# ============================================================
# ИТОГОВАЯ СВОДКА
# ============================================================

print("\n" + "=" * 70)
print("ИТОГИ ПРОВЕРКИ")
print("=" * 70)

grand_total_segments = 0
grand_total_empty = 0
grand_total_missing = 0

for corpus_name, recordings in preview_results.items():

    corpus_total = 0
    corpus_empty = 0
    corpus_missing = 0

    print(f"\n{corpus_name.upper()}:")

    for url, segments in recordings.items():

        empty = sum(not s["text"] for s in segments)
        missing = sum(not s["audio_src"] for s in segments)

        corpus_total += len(segments)
        corpus_empty += empty
        corpus_missing += missing

        print(
            f"  {url.split('/')[-1]}: "
            f"{len(segments)} сегментов, "
            f"{empty} пустых текстов, "
            f"{missing} без WAV"
        )

    print(
        f"  ИТОГО: {corpus_total} сегментов, "
        f"{corpus_empty} пустых текстов, "
        f"{corpus_missing} без WAV"
    )

    grand_total_segments += corpus_total
    grand_total_empty += corpus_empty
    grand_total_missing += corpus_missing


# ============================================================
# ОБЩАЯ СТАТИСТИКА
# ============================================================

print("\n" + "=" * 70)
print("ОБЩАЯ СТАТИСТИКА")
print("=" * 70)

print("Проверено корпусов:", len(preview_results))
print(
    "Проверено записей:",
    sum(len(recordings) for recordings in preview_results.values())
)
print("Всего сегментов:", grand_total_segments)
print("Пустых текстов:", grand_total_empty)
print("Сегментов без WAV:", grand_total_missing)

print("\nПроверка завершена.")

Все записи:   0%|          | 0/3 [00:00<?, ?запись/s]


КОРПУС: PYOZA


WAV Moseyevo1.html:   0%|          | 0/643 [00:00<?, ?сегм./s]


Запись: Moseyevo1.html
Всего audio в HTML: 643
Извлечено сегментов: 643
Говорящие: Counter({'GLA1926': 521, 'Interviewer': 122})

Первые сегменты:

--- Сегмент 1 ---
ID: Moseyevo1-5689-16932
Speaker: GLA1926
WAV: https://lingconlab.ru/pyoza/OUT/Moseyevo1-5689-16932.wav
TEXT: [] ходили если, например, есть женихи да невесты, дак выходили, уж здесь сколько сходилось дак; у нас этто хоть и деревня-то другая, дак у нас всё род-то близко. ♪

--- Сегмент 2 ---
ID: Moseyevo1-17715-26598
Speaker: GLA1926
WAV: https://lingconlab.ru/pyoza/OUT/Moseyevo1-17715-26598.wav
TEXT: Ну вот были дедом-то троюродным, а здесь много троюродных-то своих да двоюродны-то есть, наженились а не только что. ♪

--- Сегмент 3 ---
ID: Moseyevo1-28335-30724
Speaker: GLA1926
WAV: https://lingconlab.ru/pyoza/OUT/Moseyevo1-28335-30724.wav
TEXT: У меня тоже семеро детей-то вот да куды. ♪

--- Сегмент 4 ---
ID: Moseyevo1-32134-33516
Speaker: Interviewer
WAV: https://lingconlab.ru/pyoza/OUT/Moseyevo1-32134-33516.wav
TEXT: 

WAV Moseyevo2.html:   0%|          | 0/732 [00:00<?, ?сегм./s]


Запись: Moseyevo2.html
Всего audio в HTML: 732
Извлечено сегментов: 732
Говорящие: Counter({'MAN1910': 466, 'Interviewer': 266})

Первые сегменты:

--- Сегмент 1 ---
ID: Moseyevo2-4580-13975
Speaker: MAN1910
WAV: https://lingconlab.ru/pyoza/OUT/Moseyevo2-4580-13975.wav
TEXT: Камень возьмут, и рукой перешибит через реку из ручей, перешибет через реку, а сейчас вот такая река сделалась. ♪

--- Сегмент 2 ---
ID: Moseyevo2-13975-19935
Speaker: MAN1910
WAV: https://lingconlab.ru/pyoza/OUT/Moseyevo2-13975-19935.wav
TEXT: Потому что вот, видишь, идёт река-то, идёт, заворот. Заворот, видишь? ♪

--- Сегмент 3 ---
ID: Moseyevo2-19794-20535
Speaker: Interviewer
WAV: https://lingconlab.ru/pyoza/OUT/Moseyevo2-19794-20535.wav
TEXT: Да. ♪

--- Сегмент 4 ---
ID: Moseyevo2-20514-25352
Speaker: MAN1910
WAV: https://lingconlab.ru/pyoza/OUT/Moseyevo2-20514-25352.wav
TEXT: Вот подрезает, да подрезает, да подрезает да, деревню всю унесло, все переехали. ♪

--- Сегмент 5 ---
ID: Moseyevo2-25501-26232
Speake

WAV Moseyevo3.html:   0%|          | 0/791 [00:00<?, ?сегм./s]


Запись: Moseyevo3.html
Всего audio в HTML: 791
Извлечено сегментов: 791
Говорящие: Counter({'MAN1910': 499, 'Interviewer': 292})

Первые сегменты:

--- Сегмент 1 ---
ID: Moseyevo3-4460-5900
Speaker: MAN1910
WAV: https://lingconlab.ru/pyoza/OUT/Moseyevo3-4460-5900.wav
TEXT: =это всё делали. ♪

--- Сегмент 2 ---
ID: Moseyevo3-5900-10891
Speaker: MAN1910
WAV: https://lingconlab.ru/pyoza/OUT/Moseyevo3-5900-10891.wav
TEXT: Какие работы были в колхозе, всякие работы, всё время на работы, всё время на работы. ♪

--- Сегмент 3 ---
ID: Moseyevo3-11049-13700
Speaker: Interviewer
WAV: https://lingconlab.ru/pyoza/OUT/Moseyevo3-11049-13700.wav
TEXT: А в транспорте вы ходили, нет, не приходилось? ♪

--- Сегмент 4 ---
ID: Moseyevo3-13504-21532
Speaker: MAN1910
WAV: https://lingconlab.ru/pyoza/OUT/Moseyevo3-13504-21532.wav
TEXT: В транспорте-то я ходила такое время дак, до Мезе= до Доро= до Мезени ездила, раза два, да с лодкой плыла, ♪

--- Сегмент 5 ---
ID: Moseyevo3-21532-30732
Speaker: MAN1910
WAV

## 5. Обработать все записи

In [ ]:
import sys, csv, os, time
from pathlib import Path
from urllib.parse import urlparse
from urllib.parse import urlparse, unquote
import re

out_root = Path(OUTPUT_DIR)
out_root.mkdir(parents=True, exist_ok=True)
log_path = out_root / 'download.log'
print(f'[log] WAV log: {log_path}')

# Ключ создаётся самим сборщиком один раз и сохраняется на Google Drive.
all_metadata_rows = []
failed_recordings = []
seen_recording_ids = set()

urls_to_process = page_urls[:MAX_RECORDINGS] if MAX_RECORDINGS else page_urls
print(f'Будет обработано страниц: {len(urls_to_process)}')

for i, url in enumerate(urls_to_process, start=1):
    filename = Path(unquote(urlparse(url).path)).name
    recording_id = Path(filename).stem
    if not re.fullmatch(r'[\w-]+', recording_id, flags=re.UNICODE):
        print(f'[skip] Непонятный ID: {url}')
        continue
    if recording_id in seen_recording_ids:
        continue
    seen_recording_ids.add(recording_id)

    print(f'\n===== RECORD {i}/{len(urls_to_process)}: {recording_id} =====', flush=True)
    result_code = collector.main([
        '--url', url,
        '--recording-id', recording_id,
        '--out', str(out_root),
        '--delay', str(DELAY_BETWEEN_AUDIO_DOWNLOADS)
    ])
    print(f'[recording] {recording_id}: exit_code={result_code} ({i}/{len(urls_to_process)} records done)', flush=True)
    meta_path = out_root / 'metadata.csv'
    if meta_path.exists():
        with meta_path.open(encoding='utf-8-sig', newline='') as f:
            rows = list(csv.DictReader(f))
        all_metadata_rows.extend(rows)
    if result_code != 0:
        failed_recordings.append((recording_id, url, result_code))

# Общий metadata.csv для всего корпуса
if all_metadata_rows:
    meta_out = out_root / 'metadata.csv'
    fields = list(all_metadata_rows[0].keys())
    with meta_out.open('w', encoding='utf-8-sig', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fields)
        writer.writeheader()
        writer.writerows(all_metadata_rows)

print('\\nСборка завершена.')
print('Обработано записей:', len(seen_recording_ids))
print('Сегментов в метаданных:', len(all_metadata_rows))
print('Записей с ошибками:', len(failed_recordings))
for item in failed_recordings[:30]:
    print(' -', item)


## 6. Проверка структуры перед отправкой

Проверяем число сегментов, наличие WAV и текстовых пар. Файл `.speaker_salt` исключается из публикации.

In [20]:
from pathlib import Path
import csv

root = Path(OUTPUT_DIR)
with (root / 'metadata.csv').open(encoding='utf-8-sig', newline='') as f:
    metadata = list(csv.DictReader(f))

ok_audio = 0
missing = []
for row in metadata:
    for key in ('audio_file', 'text_file', 'text_orig_file'):
        path = root / row[key]
        if not path.exists():
            missing.append(str(path))
    if row['download_status'] == 'ok':
        ok_audio += 1

print('Всего сегментов:', len(metadata))
print('WAV со статусом ok:', ok_audio)
print('Отсутствующих файлов:', len(missing))
for m in missing[:20]:
    print('MISSING:', m)

print('Пример каталогов:')
for p in sorted(root.glob('*'))[:12]:
    print(p.name, '/' if p.is_dir() else '')


Всего сегментов: 338
WAV со статусом ok: 338
Отсутствующих файлов: 0
Пример каталогов:
.speaker_salt 
README.md 
metadata.csv 
Д84_21_Лавела_Занаволок /


## 7. README для репозитория

In [21]:
readme = f'''# Pinega speech corpus (PEZ-style segment layout)

This repository contains segment-level audio and transcription files collected
from the LingConLab Pinega Fulltexts / Annotated text pages.

## Layout

Each recording has its own directory. Inside `{ '{recording_id}No0' }`, each
time-aligned segment is represented by:
- `.wav` — audio segment
- `.txt` — whitespace-normalized transcription (lexical content unchanged)
- `_orig.txt` — transcription text extracted from the source HTML

`metadata.csv` contains segment IDs, speaker pseudonyms, start/end times and paths.
Each recording directory contains a `speaker_map.csv` with pseudonyms and
first-seen speaker order only.

## Source

https://lingconlab.ru/pinega/#!/fulltexts

## Privacy

Speaker labels are pseudonymized. The private `.speaker_salt` key is not included.
Do not publish the salt.

## Scope

No separate full-recording transcript files are created.
Please verify source terms and permissions before redistributing the materials.
'''
(root / 'README.md').write_text(readme, encoding='utf-8')
print('README written')


README written


## 8. Загрузить на Hugging Face

Загрузка идёт прямо из Google Drive. `.speaker_salt`, временные `.part` и локальные служебные файлы не отправляются.

In [ ]:
from huggingface_hub import HfApi

api = HfApi()

api.delete_folder(
    repo_id=HF_REPO_ID,
    repo_type="dataset",
    path_in_repo="",
    commit_message="Clear dataset before fresh upload"
)

print("Содержимое репозитория удалено.")

In [22]:
api.upload_folder(
    repo_id=HF_REPO_ID,
    repo_type='dataset',
    folder_path=OUTPUT_DIR,
    path_in_repo='.',
    ignore_patterns=['.speaker_salt', '*.part', '.ipynb_checkpoints'],
    commit_message='Upload Pinega annotated segments (PEZ-style)',
)
print(f'Загружено: https://huggingface.co/datasets/{HF_REPO_ID}/tree/main')


/usr/local/lib/python3.13/dist-packages/huggingface_hub/hf_api.py:11779: UserWarning: Warnings while validating metadata in README.md:
- empty or missing yaml metadata in repo card
  warnings.warn(f"Warnings while validating metadata in README.md:\n{message}")


Загружено: https://huggingface.co/datasets/katykool/NortDialect_benchmark/tree/main


## Примечания и ограничения

- Каталог Fulltexts извлекается браузером. Если сайт поменяет JavaScript-разметку, ячейка обнаружения ссылок может потребовать адаптации.
- Текущая логика предполагает, что ссылка `Annotated text` ведёт на `files_html/<recording_id>.html`, а WAV доступны по относительным путям в HTML.
- Парсер берёт аудиофрагменты, связанные с элементами `<audio>` и их временными ID. Он не создаёт полные транскрипции записей.
- `.speaker_salt` хранится в Google Drive для стабильных псевдонимов. Не удаляйте его между сессиями и не загружайте в HF.
- Перед публикацией корпуса убедитесь, что лицензия и условия LingConLab допускают скачивание и распространение аудио/транскрипций.
- Для первого запуска рекомендуется поставить `MAX_RECORDINGS = 2`, проверить качество результата, затем вернуть `None` и выполнить полный обход.